# BEN & XBEN streams: compressing and converting ensembles

This tutorial covers the plain **BEN/XBEN streams** and the whole-file conversion helpers:
`binary_ensemble.codec` (JSONL ↔ BEN ↔ XBEN) and `binary_ensemble.stream`
(`BenEncoder` / `BenDecoder`). Its companion, **Working with `.bendl` bundles**, covers the
recommended self-describing bundle format and the graph-reordering utilities.

**BEN** (Binary-Ensemble) compresses ensembles of districting plans. An ensemble is usually
stored as [JSONL](https://jsonlines.org) — one plan per line, like
`{"assignment": [...], "sample": n}` — which is simple but can balloon to tens of gigabytes.
BEN shrinks that losslessly; **XBEN** adds LZMA2 on top for archival-grade compression. See
[Formats](../concepts/formats.md) for the full picture.

## Setup: generate a small ensemble

So this tutorial is self-contained and reproducible, we *generate* a small ensemble instead
of downloading one: a short [GerryChain](https://gerrychain.readthedocs.io) ReCom chain on a
16×16 grid (256 nodes), written out as a JSONL file. `binary-ensemble` only ever sees lists
of integers, so any sampler — or any existing JSONL file — works the same way.

In [1]:
import json
import os
from functools import partial
from pathlib import Path

import networkx as nx
from gerrychain import Graph, MarkovChain, Partition, accept, constraints, updaters
from gerrychain.proposals import recom

Path("example_data").mkdir(exist_ok=True)

# A 16x16 grid with unit population and stripe districts -> a contiguous start state.
SIDE, N_DISTRICTS = 16, 4
grid = nx.grid_2d_graph(SIDE, SIDE)
grid = nx.convert_node_labels_to_integers(grid, ordering="sorted")
for node in grid.nodes:
    _row, col = divmod(node, SIDE)
    grid.nodes[node]["TOTPOP"] = 1
    grid.nodes[node]["district"] = col // (SIDE // N_DISTRICTS)

gc_graph = Graph.from_networkx(grid)
node_order = list(gc_graph.nodes)  # the order we write each assignment in
initial = Partition(
    gc_graph,
    assignment="district",
    updaters={"population": updaters.Tally("TOTPOP", alias="population")},
)
ideal = sum(initial["population"].values()) / len(initial)
chain = MarkovChain(
    proposal=partial(
        recom, pop_col="TOTPOP", pop_target=ideal, epsilon=0.05, node_repeats=2
    ),
    constraints=[constraints.contiguous],
    accept=accept.always_accept,
    initial_state=initial,
    total_steps=200,
)

with open("example_data/small_example.jsonl", "w") as f:
    for i, partition in enumerate(chain, start=1):
        assignment = (
            partition.assignment.to_series().loc[node_order].astype(int).tolist()
        )
        f.write(json.dumps({"assignment": assignment, "sample": i}) + "\n")

jsonl_size = os.path.getsize("example_data/small_example.jsonl")
print(
    f"wrote example_data/small_example.jsonl: 200 plans on {SIDE * SIDE} nodes, {jsonl_size} bytes"
)

wrote example_data/small_example.jsonl: 200 plans on 256 nodes, 159892 bytes


## Converting between file types

The `binary_ensemble.codec` helpers convert whole files in a single call.

In [2]:
from binary_ensemble.codec import (
    encode_jsonl_to_ben,
    encode_jsonl_to_xben,
    encode_ben_to_xben,
    decode_ben_to_jsonl,
    decode_xben_to_jsonl,
    decode_xben_to_ben,
)

### JSONL → BEN

The quickest format is BEN. `encode_jsonl_to_ben` reads the JSONL ensemble and writes a
compact `.ben` stream.

In [3]:
encode_jsonl_to_ben(
    in_file="example_data/small_example.jsonl",
    out_file="example_data/small_example.ben",
    overwrite=True,
)
print("BEN bytes:", os.path.getsize("example_data/small_example.ben"))

BEN bytes: 4303


By default the conversion functions refuse to overwrite an existing output file — pass
`overwrite=True` to replace it.

In [4]:
try:
    encode_jsonl_to_ben(
        in_file="example_data/small_example.jsonl",
        out_file="example_data/small_example.ben",
    )
except OSError as e:
    print("refused to overwrite:", e)

refused to overwrite: Output file example_data/small_example.ben already exists (use overwrite=True to replace).


### Encoding variants

A BEN stream is encoded with one of three **variants**, chosen with `variant=`:

- `"twodelta"` (the **default**) delta-encodes pairwise ReCom moves — ideal for ReCom chains.
- `"mkv_chain"` collapses identical consecutive plans — for full MCMC chains with rejections.
- `"standard"` stores each plan independently — a simple baseline.

Decoding **auto-detects** the variant, so you never specify it when reading. See
[Encoding variants](../concepts/variants.md).

In [5]:
for variant in ["standard", "mkv_chain", "twodelta"]:
    encode_jsonl_to_ben(
        "example_data/small_example.jsonl",
        f"example_data/small_example.{variant}.ben",
        overwrite=True,
        variant=variant,
    )
    print(
        f"{variant:>10}: {os.path.getsize(f'example_data/small_example.{variant}.ben'):>6} bytes"
    )

  standard:   8216 bytes
 mkv_chain:   8576 bytes
  twodelta:   4303 bytes


### BEN → XBEN

XBEN wraps a BEN stream in LZMA2 for much smaller files, at the cost of slower compression.
The XBEN encoders accept `n_threads` and `compression_level` (0 fastest … 9 smallest).

In [6]:
encode_ben_to_xben(
    in_file="example_data/small_example.ben",
    out_file="example_data/small_example.xben",
    overwrite=True,
    compression_level=9,
)

# You can also go straight from JSONL to XBEN in one step.
encode_jsonl_to_xben(
    in_file="example_data/small_example.jsonl",
    out_file="example_data/small_example.direct.xben",
    overwrite=True,
)

for name in ["small_example.jsonl", "small_example.ben", "small_example.xben"]:
    print(f"{name:>22}: {os.path.getsize('example_data/' + name):>7} bytes")

   small_example.jsonl:  159892 bytes
     small_example.ben:    4303 bytes
    small_example.xben:    2076 bytes


### Decoding

The decoders mirror the encoders and all take `(in_file, out_file, overwrite=False)`.

In [7]:
decode_ben_to_jsonl(
    "example_data/small_example.ben", "example_data/roundtrip.jsonl", overwrite=True
)
decode_xben_to_jsonl(
    "example_data/small_example.xben", "example_data/from_xben.jsonl", overwrite=True
)
decode_xben_to_ben(
    "example_data/small_example.xben", "example_data/from_xben.ben", overwrite=True
)
print("decoded BEN -> JSONL, XBEN -> JSONL, and XBEN -> BEN")

decoded BEN -> JSONL, XBEN -> JSONL, and XBEN -> BEN


Encoding is lossless. Decoding a BEN stream back to JSONL recovers the original plans exactly:

In [8]:
def load(path):
    with open(path) as f:
        return [json.loads(line)["assignment"] for line in f]


print(
    "round-trip identical:",
    load("example_data/small_example.jsonl") == load("example_data/roundtrip.jsonl"),
)

round-trip identical: True


## Streaming with `BenEncoder` / `BenDecoder`

When you'd rather write plans one at a time as they're produced (instead of from a JSONL
file), use `BenEncoder`. It's a context manager: write each assignment, and the stream is
flushed on exit. `BenDecoder` reads a stream back, one assignment at a time.

In [9]:
from binary_ensemble import BenEncoder, BenDecoder

plans = [[1, 1, 2, 2], [1, 2, 2, 2], [1, 1, 1, 2]]
with BenEncoder("example_data/tiny.ben", overwrite=True) as encoder:
    for plan in plans:
        encoder.write(plan)

decoder = BenDecoder("example_data/tiny.ben")
print("samples:", len(decoder))
for assignment in decoder:
    print(assignment)

samples: 3
[1, 1, 2, 2]
[1, 2, 2, 2]
[1, 1, 1, 2]


## Subsampling\n\nFor winnowing a large ensemble you rarely want every plan. `BenDecoder` can yield just a\nsubset without materializing the rest. Indices are 1-based. (How cheap a skipped sample is\ndepends on the variant: `standard` and `mkv_chain` frames are skipped wholesale, while\n`twodelta` — the default — replays the deltas between snapshots.)\n\nA decoder is reusable: call the `subsample_*` methods on the **same** `BenDecoder` as many\ntimes as you like. Each call rewinds to the start of the stream and applies the new\nselection, so there's no need to open a fresh decoder per subsample.

In [10]:
ben_file = "example_data/small_example.ben"
decoder = BenDecoder(ben_file)  # one decoder, reused for every subsample below

print(
    "indices [1, 100, 200] ->",
    [assignment[:6] for assignment in decoder.subsample_indices([1, 100, 200])],
)
print(
    "range(50, 53)         ->",
    [assignment[:6] for assignment in decoder.subsample_range(50, 53)],
)
print("every 50th            ->", sum(1 for _ in decoder.subsample_every(50)), "plans")

# The same decoder rewinds and re-selects on each call, so you can run subsamples
# repeatedly without building a new decoder:
print(
    "indices again         ->",
    [assignment[:6] for assignment in decoder.subsample_indices([1, 100, 200])],
)

indices [1, 100, 200] -> [[0, 0, 0, 0, 1, 1], [1, 1, 1, 1, 3, 3], [2, 2, 2, 2, 2, 2]]
range(50, 53)         -> [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1]]
every 50th            -> 4 plans
indices again         -> [[0, 0, 0, 0, 1, 1], [1, 1, 1, 1, 3, 3], [2, 2, 2, 2, 2, 2]]


The same methods work on an XBEN stream — pass `mode="xben"`. Reading XBEN pays a one-time
decompression startup cost, so if you'll subsample repeatedly, extract to BEN first with
`decode_xben_to_ben`.

In [11]:
for assignment in BenDecoder(
    "example_data/small_example.xben", mode="xben"
).subsample_range(1, 3):
    print(assignment[:6])

[0, 0, 0, 0, 1, 1]
[0, 0, 0, 0, 2, 2]
[0, 0, 0, 0, 2, 2]


/tmp/claude-1000/ipykernel_3711264/1859154157.py:1: UserWarning: XBEN may take a second to start decoding.
  for assignment in BenDecoder("example_data/small_example.xben", mode="xben").subsample_range(1, 3):


## Where to next

- [Working with `.bendl` bundles](using_bendl.ipynb) — the recommended self-describing
  container, plus graph reordering for much better compression.
- [Concepts](../concepts/overview.md) — the formats, variants, and how the compression works.
- [API reference](../api/index.md) — every public class and function.